# Conversational Memory

- Conversational memory allows our chatbots and agents to remember previous interactions within a conversation.

- Without conversational memory, our chatbots would only ever be able to respond to the last message they received, essentially forgetting all previous messages with each new message.

- Naturally, conversations require our chatbots to be able to respond over multiple interactions and refer to previous messages to understand the context of the conversation.


In [1]:
import os
import warnings
from pathlib import Path
from typing import Any, Generator, Iterable, Type, TypeVar

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


<br>

## LangChain's Memory Types

- LangChain versions 0.0.x consisted of various conversational memory types.

- Most of these are due for deprecation but still hold value in understanding the different approaches that we can take to building conversational memory.

- Throughout the notebook we will be referring to these older memory types and then rewriting them using the recommended RunnableWithMessageHistory class. 

- We will learn about:
    - **ConversationBufferMemory**: the simplest and most intuitive form of conversational memory, keeping track of a conversation without any additional bells and whistles.
    
    - **ConversationBufferWindowMemory**: similar to ConversationBufferMemory, but only keeps track of the last k messages.
    
    - **ConversationSummaryMemory**: rather than keeping track of the entire conversation, this memory type keeps track of a summary of the conversation.
    
    - **ConversationSummaryBufferMemory**: merges the ConversationSummaryMemory and ConversationTokenBufferMemory types.

- We'll work through each of these memory types in turn, and rewrite each one using the RunnableWithMessageHistory class.

In [4]:
from langchain_openai import ChatOpenAI

model_str: str = "google/gemini-2.0-flash-001"
model_str: str = "llama3.1:8b"  # "llama3.1:8b", gemma3n:e4b

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str,
)

In [5]:
response = local_llm.invoke("Tell me a short joke")
console.print(response)

AIMessage(
    content="Here's one:\n\nWhat do you call a fake noodle?\n\nAn impasta.",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 18,
            'prompt_tokens': 15,
            'total_tokens': 33,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-417',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--d5ffbf5d-2752-488b-bac5-cbcc4ebbb2d1-0',
    usage_metadata={
        'input_tokens': 15,
        'output_tokens': 18,
        'total_tokens': 33,
        'input_token_details': {},
        'output_token_details': {}
    }
)

### 1. ConversationBufferMemory

- ConversationBufferMemory is the simplest form of conversational memory, it is literally just a place that we store messages, and then use to feed messages into our LLM.

- Let's start with LangChain's original ConversationBufferMemory object, we are setting `return_messages=True` to return the messages as a list of ChatMessage objects — unless using a non-chat model we would always set this to True as without it the messages are passed as a direct string which can lead to unexpected behavior from chat LLMs.

In [6]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_21959/1448044083.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


- There are several ways that we can add messages to our memory, using the save_context method we can add a user query (via the input key) and the AI's response (via the output key). So, to create the following conversation:

    ```txt
    User: Hi, my name is Neidu
    AI: Hey Neidu, what's up? I'm an AI model called Zeta.

    User: I'm researching the different types of conversational memory.
    AI: That's interesting, what are some examples?

    User: I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.
    AI: That's interesting, what's the difference?

    User: Buffer memory just stores the entire conversation, right?
    AI: That makes sense, what about ConversationBufferWindowMemory?

    User: Buffer window memory stores the last k messages, dropping the rest.
    AI: Very cool!
    ```

We do:

In [7]:
memory.save_context(
    {"input": "Hi, my name is Neidu"},  # user message
    {"output": "Hey Neidu, what's up? I'm an AI model called Zeta."},  # AI response
)
memory.save_context(
    {
        "input": "I'm researching the different types of conversational memory."
    },  # user message
    {"output": "That's interesting, what are some examples?"},  # AI response
)
memory.save_context(
    {
        "input": "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."
    },  # user message
    {"output": "That's interesting, what's the difference?"},  # AI response
)
memory.save_context(
    {
        "input": "Buffer memory just stores the entire conversation, right?"
    },  # user message
    {
        "output": "That makes sense, what about ConversationBufferWindowMemory?"
    },  # AI response
)
memory.save_context(
    {
        "input": "Buffer window memory stores the last k messages, dropping the rest."
    },  # user message
    {"output": "Very cool!"},  # AI response
)

- Before using the memory, we need to load in any variables for that memory type — in this case, there are none, so we just pass an empty dictionary:

In [8]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMess

In [9]:
memory = ConversationBufferMemory(return_messages=True)

memory.chat_memory.add_user_message("Hi, my name is Neidu")
memory.chat_memory.add_ai_message("Hey Neidu, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message(
    "I'm researching the different types of conversational memory."
)
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message(
    "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."
)
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message(
    "Buffer memory just stores the entire conversation, right?"
)
memory.chat_memory.add_ai_message(
    "That makes sense, what about ConversationBufferWindowMemory?"
)
memory.chat_memory.add_user_message(
    "Buffer window memory stores the last k messages, dropping the rest."
)
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMess

<br>

- The outcome is exactly the same in either case. To pass this onto our LLM, we need to create a `ConversationChain` object — which is already deprecated in favor of the `RunnableWithMessageHistory` class, which we will cover in a moment.

In [10]:
from langchain.chains import ConversationChain

llm = local_llm
chain = ConversationChain(llm=llm, memory=memory, verbose=True)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_21959/3199812777.py:4: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(llm=llm, memory=memory, verbose=True)


In [11]:
chain.invoke({"input": "What's my name again?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kw

{'input': "What's my name again?",
 'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={},

<br>

### ConversationBufferMemory With RunnableWithMessageHistory

- As mentioned, the `ConversationBufferMemory` type is due for deprecation.

- Instead, we can use the `RunnableWithMessageHistory` class to implement the same functionality.

- When implementing `RunnableWithMessageHistory` we will use LangChain Expression Language `(LCEL)` and for this we need to define our prompt template and LLM components.

- Our llm has already been defined, so now we just define a ChatPromptTemplate object.

In [12]:
from langchain.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
)

system_prompt: str = (
    "You're a helpful assistant called Zeta. You answer user queries in a concise and informative manner."
)

prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(system_prompt),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template("{query}"),
    ]
)

# Connect the prompt_template with the LLM
pipeline = prompt_template | llm

- Our `RunnableWithMessageHistory` requires our pipeline to be wrapped in a `RunnableWithMessageHistory` object.

- This object requires a few input parameters and one of those is `get_session_history`, which requires a function that returns a ChatMessageHistory object based on a session ID.

- We define this function ourselves:

In [13]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map: dict[str, Any] = {}


def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    """This is used to get the chat history."""
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

- We also need to tell our runnable which variable name to use for the chat history (ie `history`) and which to use for the user's query (ie `query`).

In [14]:
from langchain_core.runnables.history import RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    runnable=pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
)

# Invoke the runnable
pipeline_with_history.invoke(
    {"query": "Hi, my name is Neidu"}, config={"session_id": "id_123"}
)

AIMessage(content='Nice to meet you, Neidu! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 43, 'total_tokens': 60, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'llama3.1:8b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-146', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--a978db37-35a8-4a1b-8df8-08a6de61ea98-0', usage_metadata={'input_tokens': 43, 'output_tokens': 17, 'total_tokens': 60, 'input_token_details': {}, 'output_token_details': {}})

In [15]:
console.print(
    pipeline_with_history.invoke(
        {"query": "What's my name again?"},
        config={"session_id": "id_123"},
    )
)

AIMessage(
    content='Your name is Neidu.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 8,
            'prompt_tokens': 75,
            'total_tokens': 83,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-109',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--087414d6-60f5-4b59-a23e-438da70f2a41-0',
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 8,
        'total_tokens': 83,
        'input_token_details': {},
        'output_token_details': {}
    }
)

### 2. ConversationBufferWindowMemory

- The ConversationBufferWindowMemory type is similar to ConversationBufferMemory, but only keeps track of the `last k` messages. 

- There are a few reasons why we would want to keep only the last k messages:

- More messages mean more tokens are sent with each request, more tokens increases latency and cost.

- LLMs tend to perform worse when given more tokens, making them more likely to deviate from instructions, hallucinate, or "forget" information provided to them. Conciseness is key to high performing LLMs.

- If we keep all messages we will eventually hit the LLM's context window limit, by adding a window size k we can ensure we never hit this limit.

- The buffer window solves many problems that we encounter with the standard buffer memory, while still being a very simple and intuitive form of conversational memory.

In [16]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=4, return_messages=True)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_21959/3216785012.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=4, return_messages=True)


In [17]:
memory.chat_memory.add_user_message("Hi, my name is James")
memory.chat_memory.add_ai_message("Hey James, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message(
    "I'm researching the different types of conversational memory."
)
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message(
    "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."
)
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message(
    "Buffer memory just stores the entire conversation, right?"
)
memory.chat_memory.add_ai_message(
    "That makes sense, what about ConversationBufferWindowMemory?"
)
memory.chat_memory.add_user_message(
    "Buffer window memory stores the last k messages, dropping the rest."
)
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={})]}

#### ConversationBufferWindowMemory With RunnableWithMessageHistory

- To implement this memory type using the RunnableWithMessageHistory class, we can use the same approach as before.

- We define our prompt_template and llm as before, and then wrap our pipeline in a RunnableWithMessageHistory object.

In [18]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field


class BufferWindowMessageHistory(BaseChatMessageHistory, BaseModel):
    """A chat message history that stores a fixed number of messages."""

    messages: list[BaseMessage] = Field(
        default_factory=list, description="List of messages in the history"
    )
    k: int = Field(
        default_factory=int, description="Number of messages to keep in the history"
    )

    def __init__(self, k: int) -> None:
        super().__init__(k=k)
        print(f"initializing {self.__class__.__name__} with k={k}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, keeping only the last k messages."""
        self.messages.extend(messages)
        self.messages = self.messages[-self.k :]

    def clear(self) -> None:
        """Clear the message history."""
        self.messages = []

In [19]:
chat_map: dict[str, Any] = {}


def get_chat_history(session_id: str, k: int = 4) -> BufferWindowMessageHistory:
    """This is used to get the chat history for a session with a buffer window."""

    print(f"get_chat_history for session={session_id!r} with k={k!r}")
    if session_id not in chat_map:
        chat_map[session_id] = BufferWindowMessageHistory(k=k)
    return chat_map[session_id]

In [20]:
from langchain_core.runnables import ConfigurableFieldSpec

pipeline_with_history = RunnableWithMessageHistory(
    runnable=pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="Unique identifier for the chat session",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="K",
            description="Number of messages to keep in the history",
            default=4,
        ),
    ],
)

In [21]:
# Invoke the runnable
response = pipeline_with_history.invoke(
    {"query": "Hi, my name is Neidu"}, config={"session_id": "id_k4", "k": 4}
)
console.print(response)

get_chat_history for session='id_k4' with k=4
initializing BufferWindowMessageHistory with k=4


AIMessage(
    content='Nice to meet you, Neidu! How can I assist you today?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 17,
            'prompt_tokens': 43,
            'total_tokens': 60,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-473',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--a27968ac-fc09-4754-9ae4-4c0872aeda7c-0',
    usage_metadata={
        'input_tokens': 43,
        'output_tokens': 17,
        'total_tokens': 60,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [22]:
# Clear the history for the session
chat_map["id_k4"].clear()

# Manually insert messages into the history
chat_map["id_k4"].add_user_message("Hi, my name is Neidu")
chat_map["id_k4"].add_ai_message("Hey Neidu, what's up? I'm an AI model called Zeta.")
chat_map["id_k4"].add_user_message(
    "I'm researching the different types of conversational memory."
)
chat_map["id_k4"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k4"].add_user_message(
    "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."
)
chat_map["id_k4"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k4"].add_user_message(
    "Buffer memory just stores the entire conversation, right?"
)
chat_map["id_k4"].add_ai_message(
    "That makes sense, what about ConversationBufferWindowMemory?"
)
chat_map["id_k4"].add_user_message(
    "Buffer window memory stores the last k messages, dropping the rest."
)
chat_map["id_k4"].add_ai_message("Very cool!")

# Only the last k messages should be kept in the history
console.print(chat_map["id_k4"].messages)

[
    HumanMessage(
        content='Buffer memory just stores the entire conversation, right?',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='That makes sense, what about ConversationBufferWindowMemory?',
        additional_kwargs={},
        response_metadata={}
    ),
    HumanMessage(
        content='Buffer window memory stores the last k messages, dropping the rest.',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={})
]

In [23]:
response = pipeline_with_history.invoke(
    {"query": "What's my name again?"},
    config={"session_id": "id_k4", "k": 4},
)

console.print(response)

get_chat_history for session='id_k4' with k=4


AIMessage(
    content="I don't have any information about your personal details. Our conversation just started, and I'm here 
to help with any questions you may have.",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 30,
            'prompt_tokens': 98,
            'total_tokens': 128,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-552',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--0186f379-4fcc-4c4c-b6ae-d6149669a55c-0',
    usage_metadata={
        'input_tokens': 98,
        'output_tokens': 30,
        'total_tokens': 128,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [24]:
# Let's try with a larger k value and a different session ID
response = pipeline_with_history.invoke(
    {"query": "Hello, my name is Neidu!"},
    config={"session_id": "id_k14", "k": 14},
)

console.print(response)

get_chat_history for session='id_k14' with k=14
initializing BufferWindowMessageHistory with k=14


AIMessage(
    content="Nice to meet you, Neidu! I'm Zeta, your friendly assistant. How can I help you today? Do you have any 
questions or topics you'd like to discuss?",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 39,
            'prompt_tokens': 44,
            'total_tokens': 83,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-471',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--43f505ca-6ce7-4619-9106-6fce846d94b5-0',
    usage_metadata={
        'input_tokens': 44,
        'output_tokens': 39,
        'total_tokens': 83,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [25]:
chat_map["id_k14"].add_user_message(
    "I'm researching the different types of conversational memory."
)
chat_map["id_k14"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k14"].add_user_message(
    "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."
)
chat_map["id_k14"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k14"].add_user_message(
    "Buffer memory just stores the entire conversation, right?"
)
chat_map["id_k14"].add_ai_message(
    "That makes sense, what about ConversationBufferWindowMemory?"
)
chat_map["id_k14"].add_user_message(
    "Buffer window memory stores the last k messages, dropping the rest."
)
chat_map["id_k14"].add_ai_message("Very cool!")

console.print(chat_map["id_k14"].messages)

[
    HumanMessage(content='Hello, my name is Neidu!', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content="Nice to meet you, Neidu! I'm Zeta, your friendly assistant. How can I help you today? Do you have 
any questions or topics you'd like to discuss?",
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 39,
                'prompt_tokens': 44,
                'total_tokens': 83,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_name': 'llama3.1:8b',
            'system_fingerprint': 'fp_ollama',
            'id': 'chatcmpl-471',
            'service_tier': None,
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='run--43f505ca-6ce7-4619-9106-6fce846d94b5-0',
        usage_metadata={
            'input_tokens': 44,
            'output_tokens': 39,
            'total_tokens': 83,
            'input_token_details': {},
            'output_token_details': {}
        }
    ),
    HumanMessage(
        content="I'm researching the different types of conversational memory.",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
    HumanMessage(
        content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
    HumanMessage(
        content='Buffer memory just stores the entire conversation, right?',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='That makes sense, what about ConversationBufferWindowMemory?',
        additional_kwargs={},
        response_metadata={}
    ),
    HumanMessage(
        content='Buffer window memory stores the last k messages, dropping the rest.',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={})
]

In [26]:
response = pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"session_id": "id_k14", "k": 14},
)

console.print(response)

get_chat_history for session='id_k14' with k=14


AIMessage(
    content='Your name is Neidu.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 8,
            'prompt_tokens': 218,
            'total_tokens': 226,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-881',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--44c67e88-69ea-4335-a42d-31d51806f192-0',
    usage_metadata={
        'input_tokens': 218,
        'output_tokens': 8,
        'total_tokens': 226,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [27]:
# Check the message history for the session
print(len(chat_map["id_k14"].messages))
console.print(chat_map["id_k14"].messages)

12


[
    HumanMessage(content='Hello, my name is Neidu!', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content="Nice to meet you, Neidu! I'm Zeta, your friendly assistant. How can I help you today? Do you have 
any questions or topics you'd like to discuss?",
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 39,
                'prompt_tokens': 44,
                'total_tokens': 83,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_name': 'llama3.1:8b',
            'system_fingerprint': 'fp_ollama',
            'id': 'chatcmpl-471',
            'service_tier': None,
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='run--43f505ca-6ce7-4619-9106-6fce846d94b5-0',
        usage_metadata={
            'input_tokens': 44,
            'output_tokens': 39,
            'total_tokens': 83,
            'input_token_details': {},
            'output_token_details': {}
        }
    ),
    HumanMessage(
        content="I'm researching the different types of conversational memory.",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
    HumanMessage(
        content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
    HumanMessage(
        content='Buffer memory just stores the entire conversation, right?',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='That makes sense, what about ConversationBufferWindowMemory?',
        additional_kwargs={},
        response_metadata={}
    ),
    HumanMessage(
        content='Buffer window memory stores the last k messages, dropping the rest.',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={}),
    HumanMessage(content='what is my name again?', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='Your name is Neidu.',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 8,
                'prompt_tokens': 218,
                'total_tokens': 226,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_name': 'llama3.1:8b',
            'system_fingerprint': 'fp_ollama',
            'id': 'chatcmpl-881',
            'service_tier': None,
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='run--44c67e88-69ea-4335-a42d-31d51806f192-0',
        usage_metadata={
            'input_tokens': 218,
            'output_tokens': 8,
            'total_tokens': 226,
            'input_token_details': {},
            'output_token_details': {}
        }
    )
]

<br><hr>

## 3. ConversationSummaryMemory

- This memory type keeps track of a summary of the conversation rather than the entire conversation.

- This is useful for long conversations where we don't need to keep track of the entire conversation, but we do want to keep some thread of the full conversation.

#### ConversationSummaryMemory With RunnableWithMessageHistory

- Let's implement this memory type using the `RunnableWithMessageHistory` class. 

- As with the window buffer memory, we need to define a custom implementation of the `InMemoryChatMessageHistory` class. We'll call this one `ConversationSummaryMessageHistory`

In [28]:
from langchain_core.messages import SystemMessage


class ConversationSummaryMessageHistory(BaseChatMessageHistory, BaseModel):
    """A chat message history that summarizes the conversation."""

    messages: list[BaseMessage] = Field(
        default_factory=list, description="List of messages in the history"
    )
    llm: ChatOpenAI = Field(
        default_factory=ChatOpenAI, description="LLM for summarization"
    )

    def __init__(self, llm: ChatOpenAI) -> None:
        super().__init__(llm=llm)
        print(f"initializing {self.__class__.__name__} with llm={llm.model_name!r}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history and summarize the conversation."""
        self.messages.extend(messages)
        # Construct the summary chat messages
        sys_msg: str = """Given the existing conversation summary and the new messages, 
        generate a new summary of the conversation. Ensure to maintain as much relevant 
        information as possible.
        """
        summary_prompt = ChatPromptTemplate.from_messages(
            [
                SystemMessagePromptTemplate.from_template(sys_msg),
                HumanMessagePromptTemplate.from_template(
                    "Existing conversation summary:\n{existing_summary}\n\nNew messages:\n{messages}"
                ),
            ]
        )
        # Format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary="".join([x.content for x in self.messages])
                or "No previous summary available.",
                messages=[x.content for x in messages],
            )
        )
        # Update the messages with the new summary
        self.messages = [SystemMessage(content=new_summary.content)]

    def clear(self) -> None:
        """Clear the message history."""
        self.messages = []

In [29]:
chat_map: dict[str, Any] = {}


def get_chat_history(
    session_id: str, llm: ChatOpenAI
) -> ConversationSummaryMessageHistory:
    """This is used to get the chat history for a session with conversation summary."""
    print(f"get_chat_history for session={session_id!r} with llm={llm.model_name!r}")
    # Check if the session_id already exists in the chat_map
    if session_id not in chat_map:
        chat_map[session_id] = ConversationSummaryMessageHistory(llm=llm)
    return chat_map[session_id]

In [30]:
pipeline_with_history = RunnableWithMessageHistory(
    runnable=pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="Unique identifier for the chat session",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatOpenAI,
            name="Large Language Model",
            description="The language model to use for the chat",
            default=llm,
        ),
    ],
)

In [31]:
response = pipeline_with_history.invoke(
    {"query": "Hi, my name is Neidu"},
    config={"session_id": "id_123", "llm": llm},
)

console.print(response)

get_chat_history for session='id_123' with llm='llama3.1:8b'
initializing ConversationSummaryMessageHistory with llm='llama3.1:8b'


AIMessage(
    content='Nice to meet you, Neidu! How can I assist you today?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 17,
            'prompt_tokens': 43,
            'total_tokens': 60,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-147',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--8409d193-3cc8-4751-9acf-710f1bf87dfb-0',
    usage_metadata={
        'input_tokens': 43,
        'output_tokens': 17,
        'total_tokens': 60,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [32]:
console.print(chat_map["id_123"].messages)

[
    SystemMessage(
        content="Here's a new summary of the conversation:\n\nNeidu introduced themselves and initiated a 
conversation with an assistant. The assistant responded with a friendly greeting and asked how they could be 
assisted.",
        additional_kwargs={},
        response_metadata={}
    )
]

In [33]:
response = pipeline_with_history.invoke(
    {"query": "I'm researching the different types of conversational memory."},
    config={"session_id": "id_123", "llm": llm},
)

console.print(response)

get_chat_history for session='id_123' with llm='llama3.1:8b'


AIMessage(
    content="Conversational memory refers to the ability of a conversational AI like myself, Zeta, to recall and 
respond to previous interactions or context within a conversation.\n\nThere are several types of conversational 
memory:\n\n1. **Session-based memory**: This type of memory allows me to retain information about the current 
conversation session, including user queries and responses.\n2. **Contextual memory**: I can use contextual 
information from the conversation to inform my responses, such as understanding the topic or intent behind a user's
query.\n3. **Long-term memory**: Some advanced conversational AI systems can store and retrieve knowledge from a 
large database of previously encountered conversations, allowing for more informed and personalized 
responses.\n\nThese types of memory enable me to provide more accurate and relevant responses to your 
queries.\n\nWould you like to know more about any specific type of conversational memory?",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 170,
            'prompt_tokens': 83,
            'total_tokens': 253,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-598',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--165d137a-0c4d-47f9-aacb-dd7ff8ff1b65-0',
    usage_metadata={
        'input_tokens': 83,
        'output_tokens': 170,
        'total_tokens': 253,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [34]:
console.print(chat_map["id_123"].messages)

[
    SystemMessage(
        content="Here's an updated summary of the conversation:\n\nNeidu introduced themselves and initiated a 
conversation with Zeta, a conversational AI. Zeta responded with a friendly greeting and asked how they could be 
assisted. Neidu mentioned that they are researching different types of conversational memory.\n\nZeta explained 
that conversational memory refers to the ability of a conversational AI like itself to recall and respond to 
previous interactions or context within a conversation. There are three main types of conversational memory:\n\n1. 
**Session-based memory**: Zeta can retain information about the current conversation session, including user 
queries and responses.\n2. **Contextual memory**: Zeta can use contextual information from the conversation to 
inform its responses, such as understanding the topic or intent behind a user's query.\n3. **Long-term memory**: 
Some advanced conversational AI systems can store and retrieve knowledge from a large database of previously 
encountered conversations, allowing for more informed and personalized responses.\n\nZeta offered to provide more 
information about any specific type of conversational memory if Neidu is interested.",
        additional_kwargs={},
        response_metadata={}
    )
]

In [35]:
for msg in [
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest.",
]:
    pipeline_with_history.invoke(
        {"query": msg}, config={"session_id": "id_123", "llm": llm}
    )

get_chat_history for session='id_123' with llm='llama3.1:8b'
get_chat_history for session='id_123' with llm='llama3.1:8b'
get_chat_history for session='id_123' with llm='llama3.1:8b'


In [36]:
console.print(chat_map["id_123"].messages)

[
    SystemMessage(
        content="Here's an updated summary of the conversation:\n\nNeidu introduced themselves and initiated a 
conversation with Zeta, a conversational AI. Zeta explained that conversational memory refers to the ability of a 
conversational AI like itself to recall and respond to previous interactions or context within a conversation. 
There are three main types of conversational memory: session-based memory, contextual memory, and long-term 
memory.\n\nNeidu expressed interest in session-based memory implementations, specifically ConversationBufferMemory 
and ConversationBufferWindowMemory. Zeta explained that both types store conversation history within a session, but
with different approaches:\n\n* **Conversation Buffer Memory**: stores all conversation history.\n* **Conversation 
Buffer Window Memory**: limits the amount of conversation history stored to a specific window size (k messages), 
dropping older ones.\n\nNeidu confirmed their understanding and acknowledged that buffer memory just stores the 
entire conversation. Zeta provided additional clarification on Conversation Buffer Window Memory, explaining its 
benefits in managing memory usage and improving performance by preventing excessive storage of conversation 
data.\n\nThe conversation concluded with Neidu considering which type of buffer memory implementation suits their 
needs best or asking for further guidance on implementing either of these types or exploring other aspects of 
session-based memory.",
        additional_kwargs={},
        response_metadata={}
    )
]

- The information about our name has been maintained, so let's see if this is enough for our LLM to correctly recall our name.

In [37]:
response = pipeline_with_history.invoke(
    {"query": "Can you remember my name?"},
    config={"session_id": "id_123", "llm": llm},
)

console.print(response)

get_chat_history for session='id_123' with llm='llama3.1:8b'


AIMessage(
    content="Your name is Neidu. I'm Zeta, your conversational AI assistant. I have a conversational memory that 
allows me to recall and respond to previous interactions within our conversation. If you'd like to continue 
discussing session-based memory or any other topic, feel free to ask!",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 59,
            'prompt_tokens': 282,
            'total_tokens': 341,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-512',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--a57672e3-7318-41bb-ab23-1605310df305-0',
    usage_metadata={
        'input_tokens': 282,
        'output_tokens': 59,
        'total_tokens': 341,
        'input_token_details': {},
        'output_token_details': {}
    }
)

<br><hr>

## 4. ConversationSummaryBufferMemory

- Our final memory type acts as a combination of ConversationSummaryMemory and ConversationBufferMemory.

- It keeps the buffer for the conversation up until the previous n tokens, anything beyond that limit is summarized then dropped from the buffer. Producing something like:

```text
# ~~ a summary of previous interactions
The user named Neidu introduced himself and the AI responded, introducing itself as an AI model called Zeta.
Neidu then said he was researching the different types of conversational memory and Zeta asked for some
examples.

# ~~ the most recent messages
Human: I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.
AI: That's interesting, what's the difference?
Human: Buffer memory just stores the entire conversation
AI: That makes sense, what about ConversationBufferWindowMemory?
Human: Buffer window memory stores the last k messages, dropping the rest.
AI: Very cool!
```

### ConversationSummaryBufferMemory with RunnableWithMessageHistory

- As with the previous memory types, we will implement this memory type again using the `RunnableWithMessageHistory` class.

- In our implementation we will modify the buffer window to be based on the number of messages rather than number of tokens. This tweak will make our implementation more closely aligned with original buffer window.

- We will implement all of this via a new `ConversationSummaryBufferMessageHistory` class.

In [38]:
class ConversationaSummaryBufferMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(
        default_factory=list, description="List of messages in the history"
    )
    llm: ChatOpenAI = Field(
        default_factory=ChatOpenAI, description="LLM for summarization"
    )
    k: int = Field(
        default_factory=int, description="Number of messages to keep in the history"
    )

    def __init__(self, llm: ChatOpenAI, k: int) -> None:
        super().__init__(llm=llm, k=k)
        print(
            f"initializing {self.__class__.__name__} with llm={llm.model_name!r} and k={str(k)!r}"
        )

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history and summarize the conversation."""
        existing_summary: SystemMessage | None = None
        old_messages: list[BaseMessage] | None = None

        # Check if a summary msg already exists
        if len(self.messages) > 0 and isinstance(self.messages[0], SystemMessage):
            print(">> Found existing summary")
            # Extract the existing summary (first message)
            existing_summary: BaseMessage = self.messages.pop(0)
        # Otherwise, create a new summary
        self.messages.extend(messages)
        # Check if we have more than k msgs
        if len(self.messages) > self.k:
            print(
                f">> Found {len(self.messages)} messages, dropping the oldest {len(self.messages) - self.k} messages"
            )
            old_messages = self.messages[: -self.k]
            self.messages = self.messages[-self.k :]

        if old_messages is None:
            print(">> No old messages to summarize")
            return

        # Construct the summary chat messages
        summary_prompt = ChatPromptTemplate.from_messages(
            [
                SystemMessagePromptTemplate.from_template(
                    "Given the existing conversation summary and the new messages, "
                    "generate a new summary of the conversation. Ensure to maintain "
                    "as much relevant information as possible."
                ),
                HumanMessagePromptTemplate.from_template(
                    "Existing conversation summary:\n{existing_summary}\n\nNew messages:\n{old_messages}"
                ),
            ]
        )
        # Format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=(
                    existing_summary.content
                    if existing_summary
                    else "No previous summary available."
                ),
                old_messages=[x.content for x in old_messages],
            )
        )
        print(f">> New summary: {new_summary.content}")
        # Prepend the new summary to the messages
        self.messages.insert(0, SystemMessage(content=new_summary.content))

    def clear(self) -> None:
        """Clear the message history."""
        self.messages = []

In [39]:
chat_map: dict[str, Any] = {}


def get_chat_history(
    session_id: str, llm: ChatOpenAI, k: int = 4
) -> ConversationaSummaryBufferMessageHistory:
    """This is used to get the chat history for a session with conversation summary."""
    print(
        f"get_chat_history for session={session_id!r} with llm={llm.model_name!r} and k={str(k)!r}"
    )
    if session_id not in chat_map:
        chat_map[session_id] = ConversationaSummaryBufferMessageHistory(llm=llm, k=k)
    return chat_map[session_id]


pipeline_with_history = RunnableWithMessageHistory(
    runnable=pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="Unique identifier for the chat session",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatOpenAI,
            name="Large Language Model",
            description="The language model to use for the chat",
            default=llm,
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="K",
            description="Number of messages to keep in the history",
            default=4,
        ),
    ],
)

In [40]:
response = pipeline_with_history.invoke(
    {"query": "Hi, my name is Neidu"},
    config={"session_id": "id_123", "llm": llm, "k": 4},
)
console.print(chat_map["id_123"].messages)

get_chat_history for session='id_123' with llm='llama3.1:8b' and k='4'
initializing ConversationaSummaryBufferMessageHistory with llm='llama3.1:8b' and k='4'
>> No old messages to summarize


[
    HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='Nice to meet you, Neidu! How can I assist you today?',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 17,
                'prompt_tokens': 43,
                'total_tokens': 60,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_name': 'llama3.1:8b',
            'system_fingerprint': 'fp_ollama',
            'id': 'chatcmpl-652',
            'service_tier': None,
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='run--8cccaabb-0053-4e28-af9b-ecf7e5c494a0-0',
        usage_metadata={
            'input_tokens': 43,
            'output_tokens': 17,
            'total_tokens': 60,
            'input_token_details': {},
            'output_token_details': {}
        }
    )
]

In [41]:
for i, msg in enumerate(
    [
        "I'm researching the different types of conversational memory.",
        "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
        "Buffer memory just stores the entire conversation",
        "Buffer window memory stores the last k messages, dropping the rest.",
    ]
):
    print(f"---\nMessage {i+1}\n---\n")
    pipeline_with_history.invoke(
        {"query": msg}, config={"session_id": "id_123", "llm": llm, "k": 4}
    )

---
Message 1
---

get_chat_history for session='id_123' with llm='llama3.1:8b' and k='4'
>> No old messages to summarize
---
Message 2
---

get_chat_history for session='id_123' with llm='llama3.1:8b' and k='4'
>> Found 6 messages, dropping the oldest 2 messages
>> New summary: Conversation Summary:

The conversation has just begun with a greeting from Neidu. The other person responded with a friendly introduction and asked how they could assist Neidu.
---
Message 3
---

get_chat_history for session='id_123' with llm='llama3.1:8b' and k='4'
>> Found existing summary
>> Found 6 messages, dropping the oldest 2 messages
>> New summary: Conversation Summary:

Neidu initiated the conversation by greeting the other person and asking for their assistance. The other person responded with a friendly introduction and offered to help Neidu. Neidu then shifted the topic to research on conversational memory, specifically asking which type of conversational memory they would like to learn more abou